In [2]:
import numpy as np  # Importamos NumPy para manejar arreglos y operaciones matemáticas

# ===== CORPUS =====
# Este es el texto con el que el modelo va a aprender (código Python en este caso)
CORPUS = r'''
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def distancia_origen(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

for i in range(10):
    print(i, fibonacci(i))
'''

# ===== VOCABULARIO =====
# Aquí obtenemos todos los caracteres únicos del corpus
chars = sorted(set(CORPUS))

# Creamos diccionario: caracter -> número
stoi = {ch: i for i, ch in enumerate(chars)}

# Diccionario inverso: número -> caracter
itos = {i: ch for ch, i in stoi.items()}

# Tamaño del vocabulario
VOCAB_SIZE = len(chars)

# Convierte texto a números
def encode(s: str):
    return [stoi[c] for c in s]

# Convierte números a texto
def decode(ids):
    return "".join(itos[i] for i in ids)

print("VOCAB_SIZE:", VOCAB_SIZE, "| len(CORPUS):", len(CORPUS))

# ===== DATASET =====
# Convertimos todo el texto a números
SEQ = np.array(encode(CORPUS), dtype=np.int64)

block_size = 32  # longitud de cada secuencia
X_rows, Y_rows = [], []

# Generamos pares (entrada, objetivo)
# X = secuencia actual
# Y = misma secuencia pero corrida 1 carácter adelante
for i in range(0, len(SEQ) - block_size):
    X_rows.append(SEQ[i : i + block_size])
    Y_rows.append(SEQ[i + 1 : i + 1 + block_size])

# Convertimos listas a arrays
X_np = np.stack(X_rows)
Y_np = np.stack(Y_rows)

print("X_np:", X_np.shape, "Y_np:", Y_np.shape)

# ===== INICIALIZACIÓN =====
rng = np.random.default_rng(42)  # generador aleatorio
H, d_emb = 64, 48  # tamaño de la RNN y embeddings

scale_e = 0.02
scale_rnn = (1.0 / np.sqrt(H + d_emb))

# Inicializamos los parámetros del modelo
def init_params(V, H, d, rng):
    E = rng.normal(0, scale_e, size=(V, d))  # embeddings
    Wxh = rng.normal(0, scale_rnn, size=(H, d))  # entrada -> hidden
    Whh = rng.normal(0, scale_rnn, size=(H, H))  # hidden -> hidden
    bh = np.zeros(H)  # bias hidden
    Why = rng.normal(0, scale_rnn, size=(V, H))  # hidden -> salida
    by = np.zeros(V)  # bias salida
    return {"E": E, "Wxh": Wxh, "Whh": Whh, "bh": bh, "Why": Why, "by": by}

params_np = init_params(VOCAB_SIZE, H, d_emb, rng)
print({k: v.shape for k, v in params_np.items()})

# ===== FORWARD =====
# Aquí el modelo procesa una secuencia paso a paso
def forward_sequence(x_indices, p, h0=None):
    T = len(x_indices)
    H = p["bh"].shape[0]

    # estado oculto inicial
    if h0 is None:
        h0 = np.zeros(H)

    # listas para guardar valores intermedios
    h_list, pre_list, emb_list, logits_list = [], [], [], []
    h_prev = h0.copy()

    # recorremos cada paso de tiempo
    for t in range(T):
        idx = int(x_indices[t])  # índice del caracter
        emb = p["E"][idx].copy()  # embedding del caracter

        # cálculo de la RNN
        pre = p["Wxh"] @ emb + p["Whh"] @ h_prev + p["bh"]
        h = np.tanh(pre)  # activación

        # salida (logits)
        logits = p["Why"] @ h + p["by"]

        # guardamos todo para backward
        emb_list.append(emb)
        pre_list.append(pre)
        h_list.append(h)
        logits_list.append(logits)

        h_prev = h  # actualizamos estado

    logits = np.stack(logits_list, axis=0)
    return logits, {"h": h_list, "pre": pre_list, "emb": emb_list, "x": x_indices}

# ===== LOSS =====
# Softmax fila por fila
def softmax_rows(z):
    z = z - z.max(axis=-1, keepdims=True)  # estabilidad numérica
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

# Entropía cruzada
def cross_entropy_from_logits(logits, targets):
    probs = softmax_rows(logits)
    T = logits.shape[0]
    nll = 0.0

    # sumamos pérdida por cada paso
    for t in range(T):
        nll -= np.log(probs[t, int(targets[t])] + 1e-12)

    return nll / T

# ===== BACKWARD =====
# Backpropagation Through Time (BPTT)
def backward_sequence(targets, p, cache):
    T = len(targets)
    V, H = p["Why"].shape

    # recalculamos probabilidades
    probs = softmax_rows(np.stack([p["Why"] @ cache["h"][t] + p["by"] for t in range(T)]))

    d_logits = probs.copy()

    # derivada de cross-entropy
    for t in range(T):
        d_logits[t, int(targets[t])] -= 1.0

    d_logits /= T

    # inicializamos gradientes
    g = {k: np.zeros_like(v) for k, v in p.items()}
    dh_next = np.zeros(H)

    # recorremos hacia atrás
    for t in range(T - 1, -1, -1):
        h = cache["h"][t]
        pre = cache["pre"][t]
        emb = cache["emb"][t]
        xi = int(cache["x"][t])

        dh = p["Why"].T @ d_logits[t] + dh_next
        dpre = (1.0 - np.tanh(pre) ** 2) * dh  # derivada tanh

        # acumulamos gradientes
        g["Why"] += np.outer(d_logits[t], h)
        g["by"] += d_logits[t]
        g["Wxh"] += np.outer(dpre, emb)
        g["Whh"] += np.outer(dpre, cache["h"][t - 1] if t > 0 else np.zeros(H))
        g["bh"] += dpre

        # embedding
        g["E"][xi] += p["Wxh"].T @ dpre

        dh_next = p["Whh"].T @ dpre

    return g

# ===== SGD =====
# Actualiza parámetros
def sgd_step(p, g, lr):
    for k in p:
        p[k] -= lr * g[k]

# ===== TEST =====
# Probamos que el modelo funcione
probe_x = X_np[0]
probe_y = Y_np[0]

logits0, cache0 = forward_sequence(probe_x, params_np)
loss0 = cross_entropy_from_logits(logits0, probe_y)
print("loss inicial:", float(loss0))

# un paso de entrenamiento
g0 = backward_sequence(probe_y, params_np, cache0)
params_test = {k: v.copy() for k, v in params_np.items()}
sgd_step(params_test, g0, lr=0.05)

logits1, _ = forward_sequence(probe_x, params_test)
loss1 = cross_entropy_from_logits(logits1, probe_y)
print("loss después de 1 paso:", float(loss1))

# ===== TRAIN =====
N = X_np.shape[0]
batch_size = 8
epochs = 80
lr = 0.02

rng = np.random.default_rng(7)

# entrenamiento por épocas
for epoch in range(1, epochs + 1):
    perm = rng.permutation(N)
    epoch_loss = 0.0
    n_batches = 0

    for start in range(0, N, batch_size):
        batch_idx = perm[start : start + batch_size]

        # acumulador de gradientes
        g_acc = {k: np.zeros_like(v) for k, v in params_np.items()}
        batch_loss = 0.0
        count = 0

        for j in batch_idx:
            x_seq = X_np[j]
            y_seq = Y_np[j]

            logits, cache = forward_sequence(x_seq, params_np)
            batch_loss += cross_entropy_from_logits(logits, y_seq)

            g = backward_sequence(y_seq, params_np, cache)

            for k in g_acc:
                g_acc[k] += g[k]

            count += 1

        # promedio del batch
        batch_loss /= max(count, 1)
        epoch_loss += batch_loss
        n_batches += 1

        for k in g_acc:
            g_acc[k] /= max(count, 1)

        # actualizamos parámetros
        sgd_step(params_np, g_acc, lr)

    if epoch == 1 or epoch % 20 == 0:
        print(f"epoch {epoch:3d} | pérdida ~ {epoch_loss / max(n_batches,1):.4f}")

print("Entrenamiento terminado.")

# ===== GENERACIÓN =====
# selecciona siguiente caracter
def sample_next(logits, temperature=1.0, rng=None):
    rng = rng or np.random.default_rng()

    z = logits / max(temperature, 1e-6)
    z = z - z.max()

    p = np.exp(z)
    p = p / p.sum()

    return int(rng.choice(len(p), p=p))

# genera texto
def complete_numpy(prompt, max_new=120, temperature=0.8, rng=None):
    rng = rng or np.random.default_rng(123)

    ids = list(encode(prompt))

    for _ in range(max_new):
        x_arr = np.array(ids[-block_size:], dtype=np.int64)

        # padding si la secuencia es corta
        if len(x_arr) < block_size:
            pad = np.full(block_size - len(x_arr), ids[0], dtype=np.int64)
            x_arr = np.concatenate([pad, x_arr])

        logits, _ = forward_sequence(x_arr, params_np)

        next_id = sample_next(logits[-1], temperature=temperature, rng=rng)
        ids.append(next_id)

    return decode(ids)

# ===== PROBAR =====
txt = complete_numpy("def fac", max_new=100, temperature=0.75)

print("\n=== TEXTO GENERADO ===\n")
print(txt[:400])

VOCAB_SIZE: 36 | len(CORPUS): 391
X_np: (359, 32) Y_np: (359, 32)
{'E': (36, 48), 'Wxh': (64, 48), 'Whh': (64, 64), 'bh': (64,), 'Why': (36, 64), 'by': (36,)}
loss inicial: 3.586872477065522
loss después de 1 paso: 3.575820814713167
epoch   1 | pérdida ~ 3.4601
epoch  20 | pérdida ~ 2.6543
epoch  40 | pérdida ~ 2.0983
epoch  60 | pérdida ~ 1.4455
epoch  80 | pérdida ~ 1.0234
Entrenamiento terminado.

=== TEXTO GENERADO ===

def fact):
 ** f b i, rsngl(g):

       aelfrnin 1:
        aelf_n = 1


                    det rnin P)c


